### CONEXION DDBB OLIST

In [1]:
%pip install PyMySQL
from sqlalchemy import create_engine, text
import ssl

## CONEXION BBDD MYSQL ##
DB_USER = "nuclio"
DB_PASS = "nuclioTFM6"
DB_HOST = "nuclio.mysql.database.azure.com"
DB_NAME = "olist"

# Crear engine apuntando a la base 'olist'
engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:3306/{DB_NAME}?charset=utf8mb4",
    pool_pre_ping=True,
    connect_args={"ssl": {"cert_reqs": ssl.CERT_NONE, "check_hostname": False}} 
)

# tablas 'olist'
with engine.connect() as conn:
    tables = conn.execute(text("SHOW TABLES")).fetchall()
    tables = [row[0] for row in tables]   # convertir a lista simple de strings
    
    print("Tablas en la base 'olist':")
    for t in tables:
        print("-", t)



Note: you may need to restart the kernel to use updated packages.
Tablas en la base 'olist':
- dash_olist_categorias_resumen
- dash_olist_demorados
- dash_olist_sellers
- dash_olist_states
- dash_olist_ventas_meses
- dash_sentiment_analysis
- distribucion_pedidos
- olist_customers_dataset
- olist_geolocation_dataset
- olist_order_items_dataset
- olist_order_payments_dataset
- olist_order_reviews_dataset
- olist_orders_dataset
- olist_products_dataset
- olist_sellers_dataset
- pedidos_por_tiempo
- product_category_name_translation


In [2]:
import pandas as pd
from IPython.display import display, Markdown

# --- Cargar datos desde SQL ---
query = """
SELECT 
    pct.product_category_name_english AS product_category_name,
    i.price,
    o.order_id,
    o.order_purchase_timestamp
FROM olist_order_items_dataset i
LEFT JOIN olist_products_dataset p
    ON i.product_id = p.product_id
LEFT JOIN product_category_name_translation pct
    ON p.product_category_name = pct.product_category_name
LEFT JOIN olist_orders_dataset o
    ON i.order_id = o.order_id
WHERE o.order_status <> 'canceled'
"""
df = pd.read_sql_query(query, con=engine)

# --- Procesamiento temporal ---
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"], errors="coerce")

# Forzar tipo entero antes de filtrar
df["order_year"] = df["order_purchase_timestamp"].dt.year.astype("Int64")
df["order_month"] = df["order_purchase_timestamp"].dt.month.astype("Int64")

# Filtrar años válidos y quitar septiembre 2018 en adelante
df = df[df["order_year"].isin([2017, 2018])]
df = df[~((df["order_year"] == 2018) & (df["order_month"] >= 9))]

# --- Mapeo de nombre de mes ---
month_map = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril",
    5: "Mayo", 6: "Junio", 7: "Julio", 8: "Agosto",
    9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"
}
df["month_name"] = df["order_month"].map(month_map)

# --- Agrupar por categoría, año y mes ---
ventas_mensuales = (
    df.groupby(["product_category_name", "order_year", "order_month", "month_name"], dropna=False)
      .agg(
          TotalSales=("price", "sum"),
          OrdersQty=("order_id", "nunique")
      )
      .reset_index()
      .sort_values(by=["order_year", "order_month", "product_category_name"])
)

# --- Añadir número de mes explícito ---
ventas_mensuales["month_number"] = ventas_mensuales["order_month"]

# --- Visualización previa ---
display(Markdown("### Ventas mensuales por categoría (enero 2017 – agosto 2018)"))
display(
    ventas_mensuales[
        ["product_category_name", "order_year", "month_number", "month_name", "TotalSales", "OrdersQty"]
    ].head(20)
)

# ---  Verificación de años cargados ---
print("\n Registros por año tras filtrado:")
print(df["order_year"].value_counts().sort_index())



### Ventas mensuales por categoría (enero 2017 – agosto 2018)

,product_category_name,order_year,month_number,month_name,TotalSales,OrdersQty
0,agro_industry_and_commerce,2017,1,Enero,65.97,2
19,air_conditioning,2017,1,Enero,663.70,3
82,auto,2017,1,Enero,5168.83,29
102,baby,2017,1,Enero,6397.87,32
122,bed_bath_table,2017,1,Enero,3960.16,45
142,books_general_interest,2017,1,Enero,234.89,2
233,computers,2017,1,Enero,2199.99,1
246,computers_accessories,2017,1,Enero,3924.14,24
266,consoles_games,2017,1,Enero,5954.20,21
286,construction_tools_construction,2017,1,Enero,310.00,1



 Registros por año tras filtrado:
order_year
2017    50617
2018    61135
Name: count, dtype: Int64


In [3]:
# ---  Comprobación de meses de 2018 aún presentes ---
check_2018 = (
    df[df["order_year"] == 2018]
    .groupby("order_month")
    .size()
    .reset_index(name="num_registros")
    .sort_values("order_month")
)

print(" Meses de 2018 presentes en el DataFrame:\n")
print(check_2018)

# Validación rápida
if (check_2018["order_month"] > 9).any():
    print("\n Aún hay datos de septiembre o posteriores en 2018.")
else:
    print("\n No hay registros de septiembre 2018 ni posteriores (filtro correcto).")


 Meses de 2018 presentes en el DataFrame:

   order_month  num_registros
0            1           8173
1            2           7597
2            3           8195
3            4           7957
4            5           7898
5            6           7060
6            7           7039
7            8           7216

 No hay registros de septiembre 2018 ni posteriores (filtro correcto).


In [4]:
from sqlalchemy import text
from sqlalchemy.types import Float, Integer, String

# ---  Guardar DataFrame en la base de datos ---
table_name = "dash_olist_ventas_meses"

# ---  Eliminar tabla si existe (evita errores de duplicado) ---
with engine.begin() as conn:
    conn.execute(text(f"DROP TABLE IF EXISTS {table_name};"))

# ---  Definir tipos de columnas ---
dtype_map = {
    "product_category_name": String(100),
    "order_year": Integer(),
    "order_month": Integer(),
    "month_number": Integer(),
    "month_name": String(20),
    "TotalSales": Float(),
    "OrdersQty": Integer(),
}

# ---  Crear tabla nueva ---
ventas_mensuales.to_sql(
    name=table_name,
    con=engine,
    if_exists="fail",  # ya eliminamos antes la tabla
    index=False,
    dtype=dtype_map,
    method="multi"
)

print(f" Tabla '{table_name}' creada y cargada correctamente en la base de datos con el campo 'month_number'.")


 Tabla 'dash_olist_ventas_meses' creada y cargada correctamente en la base de datos con el campo 'month_number'.
